# Week 4 Day 4 - Deep Agents、最上位のレイヤー

いよいよスタックの最上位に到達しました。Deep Agentは、昨日出会ったcreate_agentを取り込み、あらかじめ方針の定まったハーネスの中に包み込みます。本格的で、多くのステップを必要とする作業のために、本来であれば自分で構築するようなセットアップです。

3つの機能があらかじめ組み込まれています。planningツールがあるので、agentは自分でtodoリストを書き、それをこなしていくことができます。filesystemがあるので、すべてを会話の中に保持するのではなく、作業メモを保存して読み返すことができます。そしてsub-agentsがあるので、まとまった作業を、独立したクリーンなcontextを持つヘルパーに委ねることができます。あなたは意図（intent）を与えるだけで、ハーネスが構造を与えてくれます。

## いつこれを使うべきか

1つか2つのツールで済む簡単な質問であれば、create_agentが適切な選択であり、Deep Agentはやり過ぎになります。Deep Agentが力を発揮するのは、タスクに多くのステップがあり、成果物（artifact）を生み出し、agentが自分の作業を自分で組織化することで恩恵が得られる場合です。研究とレポート作成はその典型例なので、それをこれから構築していきます。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">実行する前に</h2>
            <span style="color:#ff7800;">このラボには<code>OPENAI_API_KEY</code>と、Web検索用の<code>SERPER_API_KEY</code>が必要です。agentは、このノートブックの隣にある<code>sandbox</code>フォルダに作業内容を書き込みます。このフォルダは以下で作成します。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# まずはインポートと環境設定

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

load_dotenv(override=True)

## Web検索ツール

agentには1つのツールを与えますが、それはすでに知っているものです。Day 2でグラフを動かした、`langchain-community`の既製のSerper検索です。たった1行で、agentはWebを検索できるようになります。

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

## 計画し、検索し、書き上げるresearch agent

こんな状況を想定してみましょう。ある会社が、営業車両を電気自動車に切り替える準備を進めていて、誰かが最初に下調べをしなければなりません。公共の充電網はどの程度準備が整っているのか、そして、実際にフリート運用で頼れる充電プロバイダーはどこなのか。これは、Deep Agentに向いている高レベルなタスクの典型です。計画が必要になるほど自由度が高く、複数回分の検索を要する調査があり、成果物として文書によるブリーフィングが求められます。

agentのfilesystemを、実際の`sandbox`フォルダに向けます。これにより、agentが書き込むものは、後で開けるディスク上のファイルとして現れます。agentは自分のfilesystemを`/`として見るので、`/charging.md`に書き込んだと報告されれば、それは私たちのsandboxフォルダの中の`charging.md`として現れます。

agentは自分でtodoリストを書き、何度かWebを検索し、ブリーフィングをファイルに保存します。

In [ ]:
sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

model = ChatOpenAI(model="gpt-5.4-mini")

researcher = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=(
        "You are a research analyst. Plan your work with your todo tool, "
        "research with the search tool, and write your findings as a tidy markdown briefing to a file."
    ),
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
researcher

In [ ]:
brief = """
Our company is planning to move its sales fleet to electric vehicles.
Research the public EV charging landscape in the US: find out roughly how many public charging points there are,
and pick out two major charging networks a fleet could rely on.
Write a one page markdown briefing, with a heading and a short section for each, to the file charging.md.
"""

result = researcher.invoke({"messages": [{"role": "user", "content": brief}]})
print(result["messages"][-1].content)

どのように機能したか見てみましょう。まずは、agentが呼び出すことにしたツールです。これによって、planningとファイル書き込みの様子が分かります。それから、sandboxフォルダの中も見てみましょう。

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the agent called, in order:")
print(tools_used)

## Sub-agents: まとまった作業を委ねる

sub-agentとは、メインのagentが`task`ツールを通じて処理を委任できるヘルパーです。ヘルパーは自分自身の新しいcontextで動作し、1つの仕事をこなして、整理された結果を報告してくれます。これにより、タスクに複数の独立した部分が含まれる場合でも、メインのagentの注意を明確に保つことができます。

充電に関するブリーフィングは、フリート切り替えが実現可能かどうかという問いに答えるものでした。次に会社が問うことになるのは、どの車を買うかということであり、これは自然に独立した部分に分かれます。それぞれの候補車両は、それぞれ単独で調査できるのです。そこで、車両調査を行うsub-agentを定義し、メインのagentに、フリート向けの2つの候補について、それぞれの調査を委任させて比較させます。

In [ ]:
research_ev_instructions = """
You research one electric vehicle using the search tool and return three concise facts
that a fleet buyer would care about, such as price, range and charging.
"""

overall_instructions = """
You write comparison briefings for a company choosing electric vehicles for its sales fleet.
For each vehicle, delegate the research to your vehicle-researcher sub-agent,
then write a markdown comparison to a file, ending with a clear recommendation.
"""


research_subagent = {
    "name": "vehicle-researcher",
    "description": "Researches a single electric vehicle and returns a short list of facts about it.",
    "system_prompt": research_ev_instructions,
}

lead = create_deep_agent(
    model=model,
    tools=[search],
    system_prompt=overall_instructions,
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
lead

In [ ]:
mission = """
Compare the Tesla Model Y and the Ford Mustang Mach-E as candidates for our 100-car sales fleet.
Research each vehicle, then write a short markdown comparison with a recommendation to fleet.md.
"""

result = lead.invoke({"messages": [{"role": "user", "content": mission}]})

In [ ]:
tools_used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", []) or [])]
print("Tools the lead agent called:", tools_used)

### では、Sandboxフォルダの中の`fleet.md`を見てみましょう

## もう一つ: Agent Skills

このブリーフィングは良い出来ですが、もっと良くすることができます。

deepagentsは、Anthropic社のAgent Skillsのパターンをサポートしています。これは、Claude Codeが使っているのと同じ`SKILL.md`形式です。skillとは、単に`SKILL.md`ファイルを含むフォルダのことです。nameとdescriptionを持つYAMLのfrontmatterに続いて、markdown形式の指示が書かれています。system promptに入るのはnameとdescriptionだけで、agentはそのskillが関連していると判断したときに、自分の`read_file`ツールでファイル全体を読みます。これは段階的開示（progressive disclosure）と呼ばれ、これによって、contextを圧迫することなく、agentにskillの大きなライブラリ全体を渡すことができます。Anthropicは、[github.com/anthropics/skills](https://github.com/anthropics/skills)で、この形式のskillのライブラリを公開しており、これにはClaude自身のドキュメント作成機能を支えているものも含まれています。

sandboxの中に1つ用意してあります。[sandbox/skills/fleet-slide/SKILL.md](sandbox/skills/fleet-slide/SKILL.md)は、私たちの架空のアナリストブランドであるVoltway Researchのハウススタイルで、レコメンデーションを1枚のスライドに仕上げるためのものです。agentに渡す前に、開いて読んでみてください。

## スライドを作るsub-agent

さあ、いよいよ本番です。スライド作成用のsub-agentを定義し、lead agentが持っていない2つのものを与えます。fleet-slideスキルと、専用の`create_slide`ツールです。このツールは、`slide_kit.py`にあるスライドテンプレートをラップしており、ブランドカラー、ロゴ、レイアウトといったデザイン作業をすべて処理してくれます。そのため、sub-agentの仕事は、ブリーフィングを読み、メッセージを抽出し、ハウススタイルに従うことだけになります。sub-agentは、それぞれ独自のツール、モデル、skillを持つことができ、これが、1つのlead agentの周りに専門家チームを構築する方法です。

In [ ]:
from langchain_core.tools import tool
from slide_kit import build_slide

@tool
def create_slide(title: str, key_points: list[str], recommendation: str) -> str:
    """Create a one-slide PowerPoint in the Voltway Research house style, saved as fleet.pptx."""
    build_slide(title, key_points, recommendation, os.path.join(sandbox, "fleet.pptx"))
    return "Saved the slide to /fleet.pptx"

slide_maker = {
    "name": "slide-maker",
    "description": "Turns a finished recommendation into a one-slide PowerPoint deck.",
    "system_prompt": "You turn research recommendations into slides, following your fleet-slide skill.",
    "tools": [create_slide],
    "skills": ["/skills/"],
}

presenter = create_deep_agent(
    model=model,
    subagents=[slide_maker],
    system_prompt="You prepare research for presentation by delegating to your slide-maker sub-agent.",
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

In [ ]:
result = presenter.invoke({"messages": [{"role": "user", "content":
    "Read fleet.md and have a one-slide deck made of its recommendation."}]})
print(result["messages"][-1].content)

sandboxには、これで`fleet.pptx`が入っています。PowerPoint、Keynote、Google Slidesのいずれかで開いて、agentが最初から最後まで自分で調査し、書き上げ、デザインしたスライドを見てみましょう。今回のブランドカラーはたまたまネイビーとティールでしたが、`slide_kit.py`のテンプレートを差し替えれば、同じagentが自分の会社のカラーでプレゼンテーションを作ってくれます。

## まとめ、そしてこれから向かう先

ここまでで、自分の作業を自分で計画し、Webを検索し、ディスクにファイルを書き込み、sub-agentに処理を委任し、Agent Skillに従ってその調査結果をブランド仕様のPowerPointスライドに仕立て上げる、Deep Agentを動かしてきました。これほどの機能が、ごくわずかなコードで手に入るというのが、このハーネスが与えてくれるものです。

明日のSidekickは、意図的にcreate_agentのレイヤーに戻ります。それが、レスポンスの良いアシスタントにとって適切な抽象化のレベルだからです。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">research agentに2つ目のツールを与えてみましょう。昨日のMCPサーバーからヘッドフルなブラウザツールを追加する場合は、それらが非同期であることを忘れずに。Day 3でやったように、<code>researcher.invoke(...)</code>を<code>await researcher.ainvoke(...)</code>に切り替える必要があります。そして、sandboxフォルダを開いて、agentが途中で自分のために書いたすべてのファイルを読んでみましょう。さらに挑戦したい人向けの課題として、自分自身のskillを書いてみてください。Voltwayのものの代わりに、あなた自身のハウススタイルをスライド作成者に教える、新しいSKILL.mdフォルダです。
            </span>
        </td>
    </tr>
</table>